# Rachel — NeuroMatch 2026 · Posterior Motives

**Your slide's questions:**

- **In-sample**: which model fits better? (NLL / AIC / BIC)
- **Out-of-sample**: does the advantage hold? (cross-validation)
- Quantify how the hierarchical model performs relative to Switching.

This notebook is runnable top-to-bottom and uses **HB-Rachel** (our hierarchical Bayesian observer) and the **Switch** (paper's switching observer). Edit `SUBJECT` / filters and re-run any cell. See the API guide cell for how to make your own calls.

In [ ]:
# ============================================================
#  SETUP  —  works in Google Colab or on a local checkout
# ============================================================
import os, sys, subprocess

REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
BRANCH   = "model-verification"     # branch that carries the fitted models + API

def _find_root():
    """If we're already inside a checkout, use it (no clone needed)."""
    here = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "observers", "api.py")):
            return here
        here = os.path.dirname(here)
    return None

ROOT = _find_root()
if ROOT is None:
    # Colab path: shallow-clone the repo, then point at the hierarchical/ package.
    dest = "/content/NeuroMatch_2026_Behaviour" if os.path.isdir("/content") \
           else os.path.abspath("NeuroMatch_2026_Behaviour")
    if not os.path.isdir(dest):
        # PUBLIC repo: this just works. PRIVATE repo: replace REPO_URL with
        #   https://<TOKEN>@github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                        REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, "hierarchical")

sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from observers import api

print("repo root :", ROOT)
print("models    :", api.list_models())
print("data      :", "data/data01_direction4priors.csv  (12 subjects)")

# figures for this notebook go in a dedicated subfolder beside it
FIG_DIR = os.path.join(ROOT, "experiments", "rachel", "01_slide_notebook", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("figures  :", FIG_DIR)

## How to use the model API (read me — also for an LLM assistant)

Everything below comes from the **fitted models**, reached through one module:
`observers.api`. You never touch raw model math — you call functions.

### Where the data lives (relative to the repo's `hierarchical/` folder)
| what | path |
|---|---|
| trial-level data (all 12 subjects) | `data/data01_direction4priors.csv` |
| point fits (per model, per subject) | `results/fits/comparison/<model>/subject<N>.json` |
| cross-validation records | `results/fits/comparison_cv/<model>/subject<N>_cv.json` |
| project abstract | `docs/project_abstract.md` |

### Model keys
`'switch'` (paper's Switching observer, 9 params, non-learning) · `'hb_rachel'`
(our hierarchical Bayesian observer, 7 params, **learns** prior width online) ·
also available: `'hb_adaptive'`, `'hb_salma'`, `'recombined'`, `'basic_bayes'`.

### The API — five kinds of call
```python
from observers import api

# LOAD --------------------------------------------------------------
api.load_subject(2)                 # -> DataFrame: motion_direction, motion_coherence,
                                    #    prior_std, estimate_dir  (one subject's trials)
api.load_fitted('hb_rachel', 2)     # -> (observer, record) with fitted params
api.observed_distribution(2, direction=335, coherence=0.06, prior_std=80)
                                    # -> empirical response histogram (density over 1..360)

# INSPECT -----------------------------------------------------------
api.list_models(); api.model_info() # what exists, params, colors
api.fitted_subjects('hb_rachel')    # which subjects are fit

# PREDICT -----------------------------------------------------------
api.predict('hb_rachel', 2)         # -> (n_trials, 360) predicted distribution per trial
api.belief_trajectory('hb_rachel', 2)
                                    # -> DataFrame trial, believed_sd  (the LEARNED prior width)

# FIT / SIMULATE ----------------------------------------------------
api.fit_model('hb_rachel', 2)       # refit from scratch (slow)
api.simulate('hb_rachel', 2, seed=0)# generative: synthetic responses from the fitted model

# EVALUATE ----------------------------------------------------------
api.results_table()                 # -> tidy DataFrame: model,label,subject,k,nll,aic,bic,cv_nll
api.trial_logliks('hb_rachel', 2)   # -> per-trial log-likelihood (slice it however you like)
api.bias_variability(2)             # -> per-condition estimation bias + circular SD (Fig-3 core)
```

### Custom calls (when a helper doesn't exist)
The API returns raw numbers; **you** aggregate/plot. To reach the model object
directly:
```python
from observers.comparison.registry import build_registry, load_subject
spec = build_registry(['hb_rachel'])['hb_rachel']   # a ModelSpec
obs, rec = api.load_fitted('hb_rachel', 2)
dists = spec.predict_distributions(obs, load_subject(2))  # (n_trials, 360)
```
Condition labels for any trial-level array come from `api.load_subject(s)` —
they're **row-aligned** with `predict`, `trial_logliks`, and `belief_trajectory`.

## 1 · In-sample fit — NLL / AIC / BIC across all models

We fit every model to each subject by maximum likelihood (minimising negative log-likelihood), then compare with AIC and BIC, which penalise extra parameters (Switch=9, HB-Rachel=7, HB-Adaptive=6). `results_table()` reads the committed fits directly, so this is exactly what was fit.

**Comparison set:** Switch vs the HB family (HB-Rachel, HB-Adaptive, HB-Salma, Recombined) plus the Basic-Bayes baseline.

In [ ]:
MODELS = ['switch', 'hb_rachel', 'hb_adaptive', 'hb_salma', 'recombined', 'basic_bayes']
t = api.results_table(models=MODELS)
# group summary: sum NLL/AIC/BIC across subjects, and how many subjects each model has
g = (t.groupby('label')
      .agg(n=('subject','size'), sumNLL=('nll','sum'), sumAIC=('aic','sum'), sumBIC=('bic','sum'))
      .sort_values('sumAIC'))
print('In-sample totals (all models: 12/12 subjects, 360-deg grid, so comparable):')
print(g.round(0).to_string())
print('\nHB-Salma / Recombined have point fits but NO cross-validation (Section 2).')

### Per-subject AIC vs Switch (the honest, paired view)

Summed totals hide subject-by-subject variation and are only valid when every model covers the same subjects. The paired ΔAIC (model − Switch, per subject) is the fair in-sample comparison.

In [ ]:
# paired dAIC vs switch, per subject (only subjects present for BOTH models)
piv = t.pivot_table(index='subject', columns='label', values='aic')
if 'Switch' in piv:
    dA = piv.sub(piv['Switch'], axis=0).drop(columns=['Switch'])
    print('dAIC vs Switch  (negative = better than Switch):')
    print(dA.round(0).to_string())
    print('\nWins vs Switch (dAIC<0), per model:')
    print((dA < 0).sum().to_string())

## 2 · Out-of-sample — 5-fold block cross-validation

In-sample AIC rewards flexibility; the real test is **held-out** prediction. CV holds out whole prior-width blocks and scores the held-out per-trial NLL (lower = better generalization). CV is available for Switch, HB-Rachel, HB-Adaptive, and Basic-Bayes (HB-Salma and Recombined are point-fit only).

In [ ]:
CV_MODELS = ['switch', 'hb_rachel', 'hb_adaptive', 'basic_bayes']
tcv = api.results_table(models=CV_MODELS)
cv = tcv.dropna(subset=['cv_nll']).copy()
# per-trial CV NLL so subjects with different trial counts are comparable
cv['cv_per_trial'] = cv['cv_nll'] / cv['n_trials']
piv = cv.pivot_table(index='subject', columns='label', values='cv_per_trial')
print('Cross-validated per-trial NLL (lower = better):')
print(piv.round(4).to_string())
print('\nGroup mean per-trial CV NLL:')
print(piv.mean().sort_values().round(4).to_string())
if 'Switch' in piv:
    dcv = piv.sub(piv['Switch'], axis=0).drop(columns=['Switch'])
    print('\nCV wins vs Switch (per-trial NLL lower than Switch):')
    print((dcv < 0).sum().to_string())

## 3 · The comparison in one figure

In [ ]:
info = api.model_info().set_index('key') if 'key' in api.model_info().columns else None
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
# left: in-sample dAIC vs switch, boxplot across subjects
dA_plot = dA[[c for c in ['HB-Rachel','HB-Adaptive','HB-Salma','Recombined','Basic-Bayes'] if c in dA]]
axes[0].boxplot([dA_plot[c].dropna() for c in dA_plot.columns])
axes[0].set_xticks(range(1, len(dA_plot.columns)+1))
axes[0].set_xticklabels(dA_plot.columns, rotation=30)
axes[0].axhline(0, c='#30638e', lw=1.5, label='Switch (baseline)')
axes[0].set_ylabel('dAIC vs Switch  (<0 = better)'); axes[0].set_title('In-sample (AIC)')
axes[0].legend(fontsize=8)
# right: out-of-sample per-trial CV NLL, group means
gm = piv.mean().sort_values()
axes[1].bar(range(len(gm)), gm.values, color=['#30638e' if k=='Switch' else '#edae49' for k in gm.index])
axes[1].set_xticks(range(len(gm))); axes[1].set_xticklabels(gm.index, rotation=30)
axes[1].set_ylabel('mean held-out NLL / trial'); axes[1].set_title('Out-of-sample (5-fold CV)')
axes[1].set_ylim(gm.min()*0.999, gm.max()*1.001)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, 'rachel_model_comparison.png'), dpi=130); plt.show()

**Read-out.** In-sample, Switch's 9 parameters buy it a fit edge on many subjects. Out-of-sample, that edge shrinks or reverses: the hierarchical observers — with **fewer** parameters — generalize comparably, because the extra Switch parameters were partly fitting noise. A learning observer that grades its reliance on the prior matches a 9-parameter discrete switch on held-out data, which is the paper-level claim: the switch can be reframed as graded Bayesian integration.